# YOLO11 trained **from scratch** on COCO 2017 (Kaggle)

In [ ]:
!pip install -q -U ultralytics pycocotools

In [ ]:
import os, sys, json, time, glob, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml, torch

os.environ["WANDB_MODE"] = "disabled"

import ultralytics
from ultralytics import YOLO, settings
for k in ("wandb", "clearml", "comet", "mlflow", "neptune"):
    try:
        settings.update({k: False})
    except Exception:
        pass

n_gpu = torch.cuda.device_count()
print("torch", torch.__version__, "| ultralytics", ultralytics.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(n_gpu)])
assert n_gpu > 0, "No GPU found. Settings -> Accelerator -> GPU T4 x2"

## 1. Config

In [ ]:
# -------------------------------------------
MODEL_CFG     = "yolo11n.yaml"   # .yaml = architecture only, RANDOM weights. (try yolo11s.yaml, yolov8n.yaml ...)
IMGSZ         = 640
BATCH         = 64               # total batch (split across GPUs)
WORKERS       = 4                # Kaggle has 4 CPU cores -> data loading is the bottleneck
TIME_HOURS    = 9.0              # training budget; Ultralytics fits the LR schedule to this. None -> use EPOCHS
EPOCHS        = 300              # only used when TIME_HOURS is None
FRACTION      = 1.0              # fraction of train2017 to use (1.0 = all 118k images)
USE_BOTH_GPUS = True
QUICK_TEST    = False            # True = tiny smoke test (2% data, 3 epochs)
RESUME_CKPT   = None             # e.g. "/kaggle/input/my-run/last.pt" to continue a stopped run
RUN_NAME      = "yolo11n_scratch_coco"
EXPORT_ONNX   = False
CLEANUP       = True             # remove symlinks to /kaggle/input before the notebook finishes
# -------------------------------------------

if QUICK_TEST:
    FRACTION, EPOCHS, TIME_HOURS = 0.02, 3, None
    RUN_NAME += "_quick"

assert MODEL_CFG.endswith(".yaml"), "Use a .yaml (no pretrained weights). A .pt file would load pretrained weights!"
DEVICE = [0, 1] if (USE_BOTH_GPUS and n_gpu > 1) else 0

WORK     = Path("/kaggle/working")
COCO_DIR = WORK / "coco"          # keep 'coco' in the path: Ultralytics then treats val as official COCO (91-class ids + pycocotools)
RUNS     = WORK / "runs"
DATA_YAML = COCO_DIR / "coco_scratch.yaml"
print("device:", DEVICE, "| model:", MODEL_CFG, "| time_h:", TIME_HOURS, "| fraction:", FRACTION)

## 2. Build the YOLO-format COCO dataset

In [ ]:
def find_path(name, want_dir, max_depth=8):
    root = "/kaggle/input"
    for dp, dns, fns in os.walk(root):
        if want_dir and name in dns:
            return os.path.join(dp, name)
        if (not want_dir) and name in fns:
            return os.path.join(dp, name)
        dns[:] = [d for d in dns if d not in ("train2017", "val2017", "test2017")]
        if dp[len(root):].count(os.sep) >= max_depth:
            dns[:] = []
    return None

src_train = find_path("train2017", True)
src_val   = find_path("val2017", True)
src_test  = find_path("test2017", True)           # optional
ann_train = find_path("instances_train2017.json", False)
ann_val   = find_path("instances_val2017.json", False)
for k, v in dict(train=src_train, val=src_val, ann_train=ann_train, ann_val=ann_val).items():
    print(f"{k:10s} {v}")
assert all([src_train, src_val, ann_train, ann_val]), "COCO 2017 not found. Add the 'COCO 2017 Dataset' to the notebook (Add Data)."

In [ ]:
from ultralytics.data.converter import convert_coco

# annotations (only the instances_* files; convert_coco chokes on caption/keypoint jsons)
(COCO_DIR / "annotations").mkdir(parents=True, exist_ok=True)
for f in (ann_train, ann_val):
    link = COCO_DIR / "annotations" / Path(f).name
    if not link.exists():
        link.symlink_to(f)

# images
(COCO_DIR / "images").mkdir(exist_ok=True)
for split, src in (("train2017", src_train), ("val2017", src_val)):
    link = COCO_DIR / "images" / split
    if not link.exists():
        link.symlink_to(src, target_is_directory=True)

# labels
if not (COCO_DIR / "labels" / "train2017").exists():
    before = set(WORK.glob("coco_converted*"))
    convert_coco(labels_dir=str(COCO_DIR / "annotations"), save_dir=str(WORK / "coco_converted"),
                 use_segments=False, cls91to80=True)
    conv = [p for p in WORK.glob("coco_converted*") if p not in before][0]
    shutil.move(str(conv / "labels"), str(COCO_DIR / "labels"))
    shutil.rmtree(conv, ignore_errors=True)

# image lists (txt lists keep the images/ -> labels/ path mapping intact; yaml dir paths get resolved through symlinks)
for split in ("train2017", "val2017"):
    imgs = sorted(p for p in os.listdir(COCO_DIR / "images" / split) if p.lower().endswith(".jpg"))
    (COCO_DIR / f"{split}.txt").write_text("\n".join(str(COCO_DIR / "images" / split / p) for p in imgs) + "\n")
    print(split, len(imgs), "images |", len(list((COCO_DIR / "labels" / split).glob("*.txt"))), "label files")

COCO80 = ["person","bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light","fire hydrant",
 "stop sign","parking meter","bench","bird","cat","dog","horse","sheep","cow","elephant","bear","zebra","giraffe","backpack",
 "umbrella","handbag","tie","suitcase","frisbee","skis","snowboard","sports ball","kite","baseball bat","baseball glove",
 "skateboard","surfboard","tennis racket","bottle","wine glass","cup","fork","knife","spoon","bowl","banana","apple",
 "sandwich","orange","broccoli","carrot","hot dog","pizza","donut","cake","chair","couch","potted plant","bed",
 "dining table","toilet","tv","laptop","mouse","remote","keyboard","cell phone","microwave","oven","toaster","sink",
 "refrigerator","book","clock","vase","scissors","teddy bear","hair drier","toothbrush"]
assert len(COCO80) == 80

with open(DATA_YAML, "w") as f:
    yaml.safe_dump({"path": str(COCO_DIR), "train": "train2017.txt", "val": "val2017.txt",
                    "names": {i: n for i, n in enumerate(COCO80)}}, f, sort_keys=False)

# sanity check: label path resolves & class ids look right
from ultralytics.data.utils import img2label_paths
first = (COCO_DIR / "train2017.txt").read_text().splitlines()[0]
lbl = img2label_paths([first])[0]
print(first, "->", lbl, "| exists:", os.path.exists(lbl))
if os.path.exists(lbl):
    print(open(lbl).read()[:200])

## 3. Train from scratch

In [ ]:
if RESUME_CKPT:
    model = YOLO(RESUME_CKPT)
else:
    model = YOLO(MODEL_CFG)                      # builds the net from yaml -> random weights
    assert not getattr(model, "ckpt", None) and getattr(model, "ckpt_path", None) is None, "Model unexpectedly has loaded weights!"
print("params (M):", round(sum(p.numel() for p in model.model.parameters()) / 1e6, 2))

train_args = dict(
    data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, workers=WORKERS, device=DEVICE,
    pretrained=False,                            # explicit: no pretrained weights
    project=str(RUNS), name=RUN_NAME, exist_ok=True,
    amp=True, cache=False, plots=True, seed=0, deterministic=False,
    fraction=FRACTION, patience=100,
)
if TIME_HOURS:
    train_args["time"] = TIME_HOURS
else:
    train_args["epochs"] = EPOCHS
if RESUME_CKPT:
    train_args = dict(resume=True)

t0 = time.time()
model.train(**train_args)
print(f"Training finished in {(time.time() - t0) / 3600:.2f} h")

In [ ]:
cands = sorted(glob.glob(str(WORK / "**" / RUN_NAME / "weights" / "best.pt"), recursive=True), key=os.path.getmtime)
assert cands, "best.pt not found"
BEST_PT = Path(cands[-1])
RUN_DIR = BEST_PT.parent.parent
print("best weights:", BEST_PT)

## 4. Training curves

In [ ]:
df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = [c.strip() for c in df.columns]
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for c in ("train/box_loss", "train/cls_loss", "train/dfl_loss"):
    ax[0].plot(df["epoch"], df[c], label=c)
for c in ("val/box_loss", "val/cls_loss", "val/dfl_loss"):
    ax[1].plot(df["epoch"], df[c], label=c)
for c in ("metrics/mAP50-95(B)", "metrics/mAP50(B)", "metrics/precision(B)", "metrics/recall(B)"):
    ax[2].plot(df["epoch"], df[c], label=c.replace("metrics/", ""))
for a, t in zip(ax, ("Train loss", "Val loss", "Val metrics")):
    a.set_title(t); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(df[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(5).to_string(index=False))

## 5. Evaluate

In [ ]:
best = YOLO(str(BEST_PT))
metrics = best.val(
    data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=32, device=0,
    conf=0.001, iou=0.7, half=True, plots=True, save_json=True,
    project=str(RUNS), name="eval_val2017", exist_ok=True,
)
EVAL_DIR = Path(metrics.save_dir)

try:
    from ultralytics.utils.torch_utils import get_flops
    gflops = round(get_flops(best.model, IMGSZ), 1)
except Exception:
    gflops = None

summary = {
    "model": MODEL_CFG, "pretrained": False, "imgsz": IMGSZ,
    "params_M": round(sum(p.numel() for p in best.model.parameters()) / 1e6, 2), "GFLOPs": gflops,
    "mAP50-95": float(metrics.box.map), "mAP50": float(metrics.box.map50), "mAP75": float(metrics.box.map75),
    "precision": float(metrics.box.mp), "recall": float(metrics.box.mr), "mean_F1": float(np.mean(metrics.box.f1)),
    "speed_ms_per_img": {k: round(float(v), 2) for k, v in metrics.speed.items()},
    "epochs_trained": int(df["epoch"].max()),
}
print(json.dumps(summary, indent=2))

### Official COCO metrics

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

pred_json = EVAL_DIR / "predictions.json"
gt = COCO(str(COCO_DIR / "annotations" / "instances_val2017.json"))
dt = gt.loadRes(str(pred_json))
E = COCOeval(gt, dt, "bbox")
E.params.imgIds = sorted(int(Path(p).stem) for p in os.listdir(COCO_DIR / "images" / "val2017") if p.endswith(".jpg"))
E.evaluate(); E.accumulate(); E.summarize()

stat_names = ["AP @[.50:.95] all", "AP @.50", "AP @.75", "AP small", "AP medium", "AP large",
              "AR @1", "AR @10", "AR @100", "AR small", "AR medium", "AR large"]
coco_table = pd.DataFrame({"metric": stat_names, "value": np.round(E.stats, 4)})
summary["coco_official"] = dict(zip(stat_names, [round(float(x), 4) for x in E.stats]))
coco_table

### Per-class results

In [ ]:
names = metrics.names
pc = pd.DataFrame({
    "class": [names[i] for i in metrics.box.ap_class_index],
    "precision": metrics.box.p, "recall": metrics.box.r, "F1": metrics.box.f1,
    "AP50": metrics.box.ap50, "AP50-95": metrics.box.ap,
}).sort_values("AP50-95", ascending=False).reset_index(drop=True)
pc.to_csv(WORK / "per_class_metrics.csv", index=False)
print("Top 10:"); print(pc.head(10).round(3).to_string(index=False))
print("\nBottom 10:"); print(pc.tail(10).round(3).to_string(index=False))

plt.figure(figsize=(7, 16))
plt.barh(pc["class"][::-1], pc["AP50-95"][::-1])
plt.xlabel("AP50-95"); plt.title("Per-class AP50-95"); plt.tight_layout(); plt.show()

### Ultralytics curves

In [ ]:
for fname in ("PR_curve.png", "F1_curve.png", "confusion_matrix_normalized.png"):
    p = EVAL_DIR / fname
    if p.exists():
        plt.figure(figsize=(9, 8) if "confusion" in fname else (8, 6))
        plt.imshow(plt.imread(p)); plt.axis("off"); plt.title(fname); plt.show()

## 6. Qualitative check

In [ ]:
val_imgs = sorted(Path(COCO_DIR / "images" / "val2017").glob("*.jpg"))
sample = random.Random(0).sample(val_imgs, 8)
res = best.predict([str(p) for p in sample], conf=0.25, imgsz=IMGSZ, device=0, verbose=False)
fig, axs = plt.subplots(2, 4, figsize=(20, 9))
for a, r in zip(axs.ravel(), res):
    a.imshow(r.plot()[..., ::-1]); a.axis("off")
plt.tight_layout(); plt.show()

# optional: test2017 has no labels -> predictions only
if src_test:
    t_imgs = sorted(Path(src_test).glob("*.jpg"))[:8]
    t_res = best.predict([str(p) for p in t_imgs], conf=0.25, imgsz=IMGSZ, device=0, verbose=False)
    fig, axs = plt.subplots(2, 4, figsize=(20, 9))
    for a, r in zip(axs.ravel(), t_res):
        a.imshow(r.plot()[..., ::-1]); a.axis("off")
    plt.suptitle("test2017 (no ground truth)"); plt.tight_layout(); plt.show()

## Save outputs

In [ ]:
shutil.copy(BEST_PT, WORK / "best.pt")
last = RUN_DIR / "weights" / "last.pt"
if last.exists():
    shutil.copy(last, WORK / "last.pt")         # use as RESUME_CKPT in a new session
with open(WORK / "metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
coco_table.to_csv(WORK / "coco_official_metrics.csv", index=False)
df.to_csv(WORK / "training_history.csv", index=False)

if EXPORT_ONNX:
    print("ONNX:", best.export(format="onnx", imgsz=IMGSZ, simplify=True))

In [ ]:
# Symlinks into /kaggle/input can bloat/break the committed output -> remove them (labels + runs are kept).
if CLEANUP:
    for p in [COCO_DIR / "images" / "train2017", COCO_DIR / "images" / "val2017"]:
        if p.is_symlink():
            p.unlink()
    for p in (COCO_DIR / "annotations").glob("*.json"):
        if p.is_symlink():
            p.unlink()
    for p in (COCO_DIR / "train2017.txt", COCO_DIR / "val2017.txt"):
        p.unlink(missing_ok=True)
    print("cleaned. (Re-run section 2 to rebuild the dataset if you want to resume training in this session.)")
print(sorted(os.listdir(WORK)))